# Setup

In [1]:
# user setup
import os

# custom Kaggle login (environment variables are recommended)
os.environ["KAGGLE_USERNAME"] = "" or os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = "" or os.getenv("KAGGLE_KEY")

# spark configuration
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" or os.getenv("JAVA_HOME")
os.environ["SPARK_HOME"] = "" or os.getenv("SPARK_HOME")

# configuration
MIN_KW_FREQ = 5 # minimum frequency of keywords to be included in the profile (for the dictionary approach)
FEATURES_SIZE = 5000 # number of features, buckets of the hash (for the hashing approach)
LSH_NUM_SKETCHES = 100 # number of random hyperplanes (signatures/sketches)
LSH_BANDS = 10 # TODO: play with b and r (calculate the threshold for different values)
LSH_ROWS = 10
LSH_MAX_BUCKET_SIZE = 500 # maximum number of users/articles in the same bucket
RECOMMENDED_ARTICLES = 10 # number of articles to recommend to each user
assert LSH_NUM_SKETCHES == LSH_BANDS * LSH_ROWS, "invalid LSH_NUM_SKETCHES, must be equal to LSH_BANDS * LSH_ROWS"


In [2]:
# setup dataset
%pip install kaggle
!test -d dataset && echo "Skipping dataset download" || (kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020" && unzip -d dataset new-york-times-articles-comments-2020.zip && rm -f new-york-times-articles-comments-2020.zip 2> /dev/null)


Note: you may need to restart the kernel to use updated packages.
Skipping dataset download


In [3]:
# setup spark
!test -d spark && echo "Skipping spark download" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1 > /dev/null && rm spark.tgz)
%pip install pyspark
%pip install py4j
import pyspark
ss = (pyspark.sql.SparkSession
    .builder
    .getOrCreate())
sc = ss.sparkContext


Skipping spark download
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
csv_options = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',
    "escape": '"', # quotes " in the dataset are escaped as ""
    "multiLine": "true",
    "mode": "PERMISSIVE",
}

full_articles = ss.read.options(**csv_options).csv("dataset/nyt-articles-2020.csv")
# full_comments = ss.read.options(**csv_options).csv("dataset/nyt-comments-2020.csv")
full_comments = ss.read.options(**csv_options).csv("dataset/nyt-comments-part0.csv") # TODO: this is a subset of all comments

print(f"Number of articles: {full_articles.count()}")
print(f"Number of comments: {full_comments.count()}")
full_articles.take(1), full_comments.take(1)


Number of articles: 16787
Number of comments: 500000


([Row(newsdesk='Editorial', section='Opinion', subsection=None, material='Editorial', headline='Protect Veterans From Fraud', abstract='Congress could do much more to protect Americans who have served their country from predatory for-profit colleges.', keywords="['Veterans', 'For-Profit Schools', 'Financial Aid (Education)', 'Frauds and Swindling', 'Colleges and Universities', 'Veterans Affairs Department', 'Federal Trade Commission', 'University of Phoenix', 'Career Education Corporation']", word_count=680, pub_date=datetime.datetime(2020, 1, 1, 0, 18, 54), n_comments=186, uniqueID='nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')],
 [Row(commentID=104387472, status='approved', commentSequence=104387472, userID=60215558, userDisplayName='magicisnotreal', userLocation='earth', userTitle=None, commentBody='Here is something I think is fraudulent that vets are subject to\n If you use your VA home loan option you have to pay higher interest rates regardless of your credit rating becua

In [5]:
%pip install mmh3 # non-cryptographic hash, better distribution
# utility hash function: bytes | str -> signed int 32
def h(data):
    import mmh3
    return mmh3.hash(data)


Note: you may need to restart the kernel to use updated packages.


## RDD

In [6]:
# parse keywords column utility
def parse_keywords(row):
    import re
    if not row["keywords"]: return []
    kws = re.sub(r'\[|\]|\'|\"', '', row["keywords"].lower()).strip()
    if not kws: return []
    return [w.strip() for w in kws.split(",") if len(w.strip()) > 0]

# keep only relevant information for recommendations porpouse, convert from dataframe to RDD tuples
articles = (full_articles
            .select("uniqueID", "newsdesk", "section", "subsection", "keywords")
            .rdd
            .map(lambda row: (row["uniqueID"], row["newsdesk"], row["section"], row["subsection"], parse_keywords(row)))
            .cache())
comments = (full_comments
            .select("articleID", "userID")
            .rdd
            .map(lambda row: (row["articleID"], row["userID"]))
            .cache())
users = (comments
         .map(lambda x: x[1])
         .distinct())

print(f"Number of articles: {articles.count()}")
print(f"Number of comments: {comments.count()}")
print(f"Number of users: {users.count()}")
articles.take(1), comments.take(1)


Number of articles: 16787
Number of comments: 500000
Number of users: 91437


([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   'Editorial',
   'Opinion',
   None,
   ['veterans',
    'for-profit schools',
    'financial aid (education)',
    'frauds and swindling',
    'colleges and universities',
    'veterans affairs department',
    'federal trade commission',
    'university of phoenix',
    'career education corporation'])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558)])

# Building Profiles

## Utility Matrix

In [7]:
utility_matrix = (comments
                  .distinct() # ignore multiple comments from same user on same article
                  .cache())
comments.unpersist()

print(f"Number of entries in sparse utility matrix: {utility_matrix.count()}")
utility_matrix.take(5)


Number of entries in sparse utility matrix: 356283


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65691034),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65110053),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 66232009),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 73299044)]

## Articles Profile - via dictionary

In [8]:
# count frequencies
keywords = (articles
            .flatMap(lambda row: [(k, 1) for k in row[4]])
            .reduceByKey(lambda a, b: a + b))

# prune too rare keywords
pruned_keywords = (keywords
                   .filter(lambda x: x[1] >= MIN_KW_FREQ)
                   .map(lambda x: f"KEY {x[0]}")
                   .collect())

# extract all features
features = pruned_keywords + (articles
                              .flatMap(lambda row: [f"{f} {row[i+1].lower().strip()}" for i, f in enumerate(["NEW", "SEC", "SUB"]) if row[i+1]])
                              .distinct()
                              .collect())

# features mapping dictionary, needs to be broadcasted to every node
features_mapping = {f: i for i, f in enumerate(features)}
print(f"Extacted {len(features_mapping)} features (broadcasting ~{((100 * 4) + 4) * len(features_mapping) // 1024} KB)")
broadcast_dict = sc.broadcast(features_mapping)

# build sparse vectors for profiles (article_id, feature_index)
def build_article_profile_dict(row):
    mapping = broadcast_dict.value
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        if f not in mapping: continue
        res.append((row[0], mapping[f]))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        if k not in mapping: continue
        res.append((row[0], mapping[k]))
    return res

articles_profile_dict = (articles
                         .flatMap(build_article_profile_dict) # (article_id, feature_index)
                         .map(lambda x: (x[0], (x[1], 1))) # (article_id, (feature_index, frequency = 1))
                         .cache())
print(f"Number of entries in sparse articles profile via dictionary: {articles_profile_dict.count()}")
articles_profile_dict.take(10)


Extacted 3555 features (broadcasting ~1402 KB)
Number of entries in sparse articles profile via dictionary: 158079


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3384, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3385, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (0, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (4, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (5, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (6, 1)),
 ('nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c', (3386, 1))]

## Articles Profile - via hashing

In [9]:
# build sparse vectors for profiles (article_id, feature_hash)
def build_article_profile_hash(row):
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        res.append((row[0], h(f) % FEATURES_SIZE))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        res.append((row[0], h(k) % FEATURES_SIZE))
    return res

articles_profile_hash = (articles
                         .flatMap(build_article_profile_hash) # (article_id, feature_hash)
                         .map(lambda x: (x[0], (x[1], 1))) # (article_id, (feature_hash, frequency = 1))
                         .cache())
articles.unpersist()

print(f"Number of entries in sparse articles profile via hash: {articles_profile_hash.count()}")
articles_profile_hash.take(10)


Number of entries in sparse articles profile via hash: 185396


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (653, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2704, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2692, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1483, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (4497, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (260, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1675, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2683, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3196, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (598, 1))]

## WARNING

From now on, the operations will be **independent** of how the profiles were computed (via dictionary or hashing).
Change the content of next cell to change approach: (`articles_profile_dict` or `articles_profile_hash`)

In [10]:
# articles_profile = articles_profile_dict.cache()
articles_profile = articles_profile_hash.cache()

articles.unpersist()
articles_profile_dict.unpersist()
articles_profile_hash.unpersist()


PythonRDD[71] at RDD at PythonRDD.scala:53

## Users Profile

In [11]:
# build user profiles by joining utility matrix with article profiles, then counting feature frequencies for each user
users_profile = (utility_matrix
                 .join(articles_profile) # (article_id, (user_id, (feature_index, frequency = 1)))
                 .map(lambda x: ((x[1][0], x[1][1][0]), 1)) # ((user_id, feature_index), 1)
                 .reduceByKey(lambda a, b: a + b) # ((user_id, feature_index), frequency)
                 .map(lambda x: (x[0][0], (x[0][1], x[1]))) # (user_id, (feature_index, frequency))
                 .cache())
print(f"Number of entries in sparse users profile: {users_profile.count()}")
users_profile.take(10)


Number of entries in sparse users profile: 2801463


[(5646097, (535, 36)),
 (5646097, (2881, 36)),
 (5646097, (4221, 36)),
 (86273619, (535, 11)),
 (86273619, (2881, 11)),
 (86273619, (4221, 11)),
 (61536727, (535, 9)),
 (61536727, (2881, 9)),
 (61536727, (4221, 9)),
 (41508509, (535, 20))]

# Computing Similarity

## Profiles sketches (signatures)

In [12]:
import random

# random vectors of +1 and -1 of dimensione FEATURES_SIZE
random_hyperplanes = [[random.choice([-1, 1]) for _ in range(FEATURES_SIZE)] for _ in range(LSH_NUM_SKETCHES)]
b_hyperplanes = sc.broadcast(random_hyperplanes)
print(f"Generated {LSH_NUM_SKETCHES} random hyperplanes (broadcasting ~{(LSH_NUM_SKETCHES * FEATURES_SIZE * 4) // 1024} KB)")

def compute_sketch(row):
    item_id, sparse_features = row
    sketch = []
    for plane in b_hyperplanes.value:
        # dot product: if above or below the hyperplane
        # only considering the features that exist in the sparse vector
        dot_product = sum(weight * plane[feat] for feat, weight in sparse_features)
        # 1 if positive, 0 if negative
        sketch.append(1 if dot_product > 0 else 0)
    return (item_id, sketch)

# compute sketches ("equivalent" of signature) for users and articles
users_sketches = (users_profile # (user_id, (feature_index, frequency))
                  .groupByKey()
                  .mapValues(list) # (user_id, [(feature_index, frequency), ...])
                  .map(compute_sketch) # (user_id, sketch)
                  .cache())
articles_sketches = (articles_profile # (article_id, (feature_index, frequency = 1))
                     .groupByKey()
                     .mapValues(list) # (article_id, [(feature_index, frequency), ...])
                     .map(compute_sketch) # (article_id, sketch)
                     .cache())

print(f"Number of users sketches: {users_sketches.count()}")
print(f"Number of articles sketches: {articles_sketches.count()}")
users_sketches.take(1), articles_sketches.take(1)


Generated 100 random hyperplanes (broadcasting ~1953 KB)
Number of users sketches: 91437
Number of articles sketches: 16787


([(13120442,
   [1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   [0,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
   

## LSH buckets

In [13]:
def lsh_buckets(row):
    item_id, sketch = row
    res = []
    for band in range(LSH_BANDS):
        band_signature = tuple(sketch[band*LSH_ROWS : (band+1)*LSH_ROWS])
        bucket_hash = h(f"b{band}_{band_signature}") # include band index to avoid collisions between different bands
        res.append((bucket_hash, item_id)) # no need to differentiate between users and articles: different format and distinct rdds
    return res

users_buckets = users_sketches.flatMap(lsh_buckets) # (bucket_hash, user_id)
articles_buckets = articles_sketches.flatMap(lsh_buckets) # (bucket_hash, article_id)

users_sketches.unpersist()
articles_sketches.unpersist()

print(f"Total number of users buckets: {users_buckets.count()}")
print(f"Total number of articles buckets: {articles_buckets.count()}")

# buckets too big are not useful, too few information
valid_users_buckets = (users_buckets
                       .map(lambda x: (x[0], 1))
                       .reduceByKey(lambda a, b: a + b)
                       .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

valid_articles_buckets = (articles_buckets
                          .map(lambda x: (x[0], 1))
                          .reduceByKey(lambda a, b: a + b)
                          .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

# the join works as an intersection between keys
users_buckets = (users_buckets # (bucket_hash, user_id)
                 .join(valid_users_buckets) # (bucket_hash, (user_id, count))
                 .map(lambda x: (x[0], x[1][0])) # (bucket_hash, user_id)
                 .cache())
articles_buckets = (articles_buckets # (bucket_hash, article_id)
                    .join(valid_articles_buckets) # (bucket_hash, (article_id, count))
                    .map(lambda x: (x[0], x[1][0])) # (bucket_hash, article_id)
                    .cache())

users_sketches.unpersist()
articles_sketches.unpersist()

print(f"Valid (less than {LSH_MAX_BUCKET_SIZE} users) number of users buckets: {users_buckets.count()} ")
print(f"Valid (less than {LSH_MAX_BUCKET_SIZE} articles) number of articles buckets: {articles_buckets.count()} ")

users_buckets.take(10), articles_buckets.take(10)


Total number of users buckets: 914370
Total number of articles buckets: 167870
Valid (less than 500 users) number of users buckets: 572393 
Valid (less than 500 articles) number of articles buckets: 160923 


([(-1895813752, 79217680),
  (-1895813752, 72890248),
  (-1895813752, 23678472),
  (-1895813752, 43682836),
  (-1895813752, 37613915),
  (-1895813752, 56864683),
  (-1895813752, 65501297),
  (-1895813752, 57914321),
  (808613084, 79217680),
  (808613084, 72890248)],
 [(967667862, 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd'),
  (967667862, 'nyt://article/fdf55793-554c-55bd-9439-ac1ffcfb0ae5'),
  (967667862, 'nyt://interactive/be8b96e0-bdd8-5896-afb3-7d0264375b88'),
  (967667862, 'nyt://article/5297a042-d39f-56f0-9abe-f24a508d90b9'),
  (967667862, 'nyt://article/80b21e7a-96e2-5ba3-b7cf-6f127cdad5a6'),
  (967667862, 'nyt://article/e4f483a4-d52a-5535-ba96-4a6bdde1a7e0'),
  (967667862, 'nyt://article/ec0462f4-81d0-5292-84cb-9ae5ece8920a'),
  (967667862, 'nyt://article/035d0316-9d4b-5aa6-bf44-b4ee91e72d2e'),
  (967667862, 'nyt://article/f0510da8-1ef8-5442-a909-8af53b7dbca4'),
  (967667862, 'nyt://interactive/2cd10ef4-810e-50db-8652-612adcd687b3')])

## LSH filter

In [14]:
lsh_recommendations = (users_buckets # (bucket_hash, user_id)
                       .join(articles_buckets) # (bucket_hash, (user_id, article_id))
                       .map(lambda x: (x[1][0], x[1][1])) # (user_id, article_id)
                       .distinct() # remove matches from multiple bands
                       .subtract(utility_matrix.map(lambda x: (x[1], x[0])))
                       .cache()) # remove already commented articles

utility_matrix.unpersist()
users_buckets.unpersist()
articles_buckets.unpersist()
users_buckets.unpersist()
articles_buckets.unpersist()

print(f"Number of recommendations that passed LSH filter: {lsh_recommendations.count()}")
lsh_recommendations.take(10)


Number of recommendations that passed LSH filter: 12676584


[(50866928, 'nyt://article/8c583043-9686-5cc7-b27f-238c76858a18'),
 (62936877, 'nyt://article/a87f0359-ea91-5488-b11d-26a0fdb953b1'),
 (58792946, 'nyt://article/f1e9abec-c545-5396-9c64-5380d47fbb91'),
 (14154828, 'nyt://article/1478eddc-7f3f-5801-8416-73e296fb6504'),
 (52697676, 'nyt://article/fc8ed5c2-dfc4-5eff-a1fc-be461cfeb06a'),
 (69507922, 'nyt://article/905910fd-da24-59c9-9457-a1bd53ec010f'),
 (38603552, 'nyt://article/e4dc642c-b541-534a-bdcb-cabdd6578a1d'),
 (17620436, 'nyt://article/8bf28996-1c5e-5457-9262-0888ac95ad49'),
 (61630073, 'nyt://article/329e2ce2-ab38-5109-be34-dd179655c016'),
 (105944931, 'nyt://article/fbc1bec9-6c59-5d2c-b37c-cc915c1cc368')]

## Actual Cosine Similarity

In [15]:
# cosine similarity between sparse profiles, user_profile and article_profile should be dicts {feature_index: weight}
def cosine_similarity(user_profile, article_profile):
    dot_product = sum(user_profile.get(feat, 0) * weight for feat, weight in article_profile.items())
    user_norm = sum(weight ** 2 for weight in user_profile.values()) ** 0.5
    article_norm = sum(weight ** 2 for weight in article_profile.values()) ** 0.5
    if user_norm == 0 or article_norm == 0:
        return 0.0
    return dot_product / (user_norm * article_norm)

# calculate actual similarity by joining the profiles and applying
recommendations = (lsh_recommendations # (user_id, article_id)
                   .join(users_profile.groupByKey().mapValues(dict)) # (user_id, (article_id, {profile}))
                   .map(lambda x: (x[1][0], (x[0], x[1][1]))) # (article_id, (user_id, {user_profile}))
                   .join(articles_profile.groupByKey().mapValues(dict)) # (article_id, ((user_id, {user_profile}), {article_profile}))
                   .map(lambda x: (x[1][0][0], (x[0], cosine_similarity(x[1][0][1], x[1][1])))) # (user_id, (article_id, actual_sim))
                   .filter(lambda x: x[1][1] > 0.0)
                   .cache())

lsh_recommendations.unpersist()
users_profile.unpersist()
articles_profile.unpersist()

print(f"Number of recommendations with actual cosine similarity: {recommendations.count()}")
recommendations.take(10)


Number of recommendations with actual cosine similarity: 6448220


[(96898068,
  ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.18181818181818182)),
 (67051638,
  ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.11134044285378082)),
 (52142940,
  ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.06741998624632421)),
 (21406554,
  ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.2849014411490949)),
 (68682312,
  ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.2558408596267325)),
 (48765303,
  ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.05415303610738823)),
 (91970910,
  ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.45454545454545453)),
 (80120007,
  ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.11030082678439321)),
 (39441114,
  ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.15075567228888181)),
 (1231632,
  ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.13794014696151088))]

# Content-Based Recommendations

In [16]:
import heapq

user_recommendations = (recommendations
                        .map(lambda x: (x[0], (x[1][0], round(x[1][1], 5))))
                        .groupByKey()
                        .mapValues(lambda iter: heapq.nlargest(RECOMMENDED_ARTICLES, iter, key=lambda x: x[1]))
                        .cache())

print(f"Number of users with at least one recommendation: {user_recommendations.count()}/{users.count()}")
user_recommendations.take(10)


Number of users with at least one recommendation: 85085/91437


[(52142940,
  [('nyt://article/33725a67-4fdf-5b5d-94dc-470e798da167', 0.57307),
   ('nyt://article/ef7a634f-91f4-5817-a5aa-4046f8aba987', 0.53033),
   ('nyt://article/2d9cc541-f938-5221-b53d-8b7e697352e2', 0.52715),
   ('nyt://article/0c4884df-395e-5930-9f94-86b85416afc4', 0.5164),
   ('nyt://article/e29f84c6-fd29-5660-980b-0ad482649c0b', 0.5164),
   ('nyt://article/a28caf6e-d3d0-5105-93e1-1b4a3c868073', 0.50709),
   ('nyt://article/aa4f7721-168a-5821-95aa-c7d8b4a9139b', 0.49614),
   ('nyt://article/6b82db52-6eb7-5271-a3d4-a070d1cc7a59', 0.49497),
   ('nyt://article/9e8f52b5-ffcc-5940-8c6e-280ec1e0c04c', 0.49497),
   ('nyt://article/5648165c-3036-5f71-800d-1cd144f27b2c', 0.49497)]),
 (91970910,
  [('nyt://article/0bd15845-cd4f-5e3e-89fb-ae2c0cb2c29b', 0.45584),
   ('nyt://article/fa50e0db-81a1-575b-8fef-4f93375664c6', 0.45455),
   ('nyt://article/5fe6011c-267b-5943-b3d8-7827d98fe0ff', 0.40202),
   ('nyt://article/df0a3596-88b6-513f-9400-2f289073366e', 0.40202),
   ('nyt://article/2df71

## Human Evaluation

In [22]:
USER = users.takeSample(False, 1)[0]

# "notable" users:
# USER = 65969500 # bad results
# USER = 22564580 # policits but not trump

read = (utility_matrix
        .filter(lambda x: x[1] == USER)
        .join(full_articles.rdd.map(lambda x: (x['uniqueID'], x.asDict()))) # (article_id, (user_id, article_info))
        .map(lambda x: x[1][1]) # (article_info)
        .collect())

reccs = (user_recommendations
         .filter(lambda x: x[0] == USER)
         .map(lambda x: x[1])
         .flatMap(lambda x: x)
         .join(full_articles.rdd.map(lambda x: (x['uniqueID'], x.asDict()))) # (article_id, article_info)
         .map(lambda x: (x[1][1], x[1][0])) # (article_info, similarity)
         .collect())

print("--- ARTICLES COMMENTED ---")
for r in read:
    kws = "\n\t".join(parse_keywords(r))
    print(f"  {r['headline']}\n\t{r['newsdesk']} - {r['section']} - {r['subsection']}\n\t{kws}")

print("--- ARTICLES RECOMMENDED ---")
for r, s in sorted(reccs, key=lambda x: x[1], reverse=True):
    kws = "\n\t".join(parse_keywords(r))
    print(f"  {round(s, 5)} {r['headline']}\n\t{r['newsdesk']} - {r['section']} - {r['subsection']}\n\t{kws}")


--- ARTICLES COMMENTED ---
  A Quick Quiz to Match You With a Democratic Candidate
	U.S. - U.S. - Politics
	presidential election of 2020
  What Iowa Caucusgoers Have to Say
	U.S. - U.S. - Politics
	presidential election of 2020
	primaries and caucuses
	polls and public opinion
	iowa
--- ARTICLES RECOMMENDED ---
  0.82078 January Democratic Debate: Here Are the Key Matchups to Watch
	U.S. - U.S. - Politics
	presidential election of 2020
	democratic party
  0.69369 What Happens When the Election Results Are Contested
	U.S. - U.S. - Politics
	presidential election of 2020
	elections
	electoral college
	voter fraud (election fraud)
  0.64889 Exactly How Much Air Time Did the Trump Family Get?
	U.S. - U.S. - Politics
	presidential election of 2020
	republican national convention
	republican party
	trump
	donald j
  0.61559 Voting by Mail Is the Hot New Idea. Is There Time to Make It Work?
	Politics - U.S. - Politics
	presidential election of 2020
	absentee voting
  0.61178 Trump vs. Biden: